In [23]:
import os

os.environ["HUGGINGFACE_TOKEN"] = "xxx"

from huggingface_hub import login
login(token=os.environ["HUGGINGFACE_TOKEN"])

import pandas as pd
from tqdm import tqdm
tqdm.pandas()

import yaml

In [2]:
from transformers import BitsAndBytesConfig
import torch

model_variant = "27b-text-it"  # @param ["4b-it", "27b-it", "27b-text-it"]
model_id = f"google/medgemma-{model_variant}"

use_quantization = True  # @param {type: "boolean"}

# @markdown Set `is_thinking` to `True` to turn on thinking mode. **Note:** Thinking is supported for the 27B variants only.
is_thinking = False  # @param {type: "boolean"}

model_kwargs = dict(
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

if use_quantization:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

In [3]:
from transformers import pipeline

if "text" in model_variant:
    pipe = pipeline("text-generation", model=model_id, model_kwargs=model_kwargs)
else:
    pipe = pipeline("image-text-to-text", model=model_id, model_kwargs=model_kwargs)

# pipe.model.generation_config.do_sample = False

Loading checkpoint shards: 100%|██████████| 11/11 [11:41<00:00, 63.79s/it]
Device set to use cuda:0


In [32]:


def load_config(p):
    with open('config_instruction.yaml', 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    return config['parameters'][p]

def prompt_generation(row, p):
    if p in ['p1', 'p5']:
        prompt = f'''Clinical Note:{row['drug_adjacent']},
    
                Input Drug: {row['standard_drug']}

                Please follow the steps.
                ''' 
    elif p in ['p2', 'p4', 'p6']:
        prompt = f''' Patient Note:{row['symptom_adjacent']}
                            
                 Please follow the steps.
                '''
    elif p == 'p3':
        prompt = f'''Patient Note:{row['symptom_adjacent']},

                Input Drug:{row['standard_drug']},
                
                Please follow the steps.
                '''
    else:
        return 'prompt_id_not_recognized'
        

    role_instruction = "You are a helpful medical assistant."
    system_instruction = f"SYSTEM INSTRUCTION: think silently if needed. {load_config(p)}"
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_instruction}]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        }
    ]
    return messages
    

def calling_llm(messages):
    output = pipe(messages, max_new_tokens=5500)
    response = output[0]["generated_text"][-1]["content"]
    return response
    
def main():
    for index in ['p1', 'p2', 'p3', 'p4', 'p5', 'p6']:
        df = pd.read_csv(f'../irAE_preprocessing/data/dfs_llm/df{index[-1]}.csv')
        print(f'{index}_inferring')
        df['output'] = df.progress_apply(lambda row: calling_llm(prompt_generation(row, index)), axis=1)
        df.to_csv(f'./llm_results/llm_df{index[-1]}.csv', index=False)

In [33]:
main()

p2_inferring


100%|██████████| 20/20 [12:20<00:00, 37.02s/it]


p3_inferring


100%|██████████| 23/23 [30:22<00:00, 79.22s/it]


p4_inferring


100%|██████████| 20/20 [14:50<00:00, 44.51s/it]


p5_inferring


100%|██████████| 8/8 [05:54<00:00, 44.35s/it]


p6_inferring


100%|██████████| 20/20 [15:01<00:00, 45.10s/it]


In [3]:
prompt = "How do you differentiate bacterial from viral pneumonia?"  # @param {type: "string"}

role_instruction = "You are a helpful medical assistant."
system_instruction = f"SYSTEM INSTRUCTION: think silently if needed. {role_instruction}"
max_new_tokens = 1500

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_instruction}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": prompt}]
    }
]

In [4]:
output = pipe(messages, max_new_tokens=max_new_tokens) #8192
response = output[0]["generated_text"][-1]["content"]

In [6]:
response

"<unused94>thought\nHere's a thinking process to construct the explanation differentiating bacterial and viral pneumonia:\n\n1.  **Understand the Goal:** The user wants to know how to tell the difference between bacterial pneumonia and viral pneumonia. This requires explaining the underlying causes, symptoms, diagnostic methods, and treatments for each. The key is to highlight the *differences* and *similarities*.\n\n2.  **Define Pneumonia:** Start with a basic definition of pneumonia itself – inflammation of the air sacs (alveoli) in the lungs, often caused by infection. This sets the context.\n\n3.  **Identify the Main Difference:** The core difference is the *cause*. Bacterial pneumonia is caused by bacteria, viral pneumonia by viruses. This is the fundamental distinction.\n\n4.  **Expand on Causes:**\n    *   **Bacterial:** Name common culprits (e.g., *Streptococcus pneumoniae* is the most frequent). Mention that it can occur on its own or follow a viral infection (secondary bacter